<a href="https://colab.research.google.com/github/Tahmidro/phitronNotebooks/blob/main/module_14_logistic_regression_implementation_using_sklearn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [71]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder, MinMaxScaler,OrdinalEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix,precision_score,recall_score,f1_score
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [57]:
df=pd.read_csv("titanic_data_updated.csv")
df.sample(6)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
883,884,no,second,"Banfield, Mr. Frederick James",male,28.0,0,0,C.A./SOTON 34068,10.5000,NaN,S
213,214,no,second,"Givard, Mr. Hans Kristensen",male,30.0,0,0,250646,13.0000,NaN,S
613,614,no,third,"Horgan, Mr. John",male,NaN,0,0,370377,7.7500,NaN,Q
156,157,yes,third,"Gilnagh, Miss. Katherine ""Katie""",female,16.0,0,0,35851,7.7333,NaN,Q
12,13,no,third,"Saundercock, Mr. William Henry",male,20.0,0,0,A/5. 2151,8.0500,NaN,S
612,613,yes,third,"Murphy, Miss. Margaret Jane",female,NaN,1,0,367230,15.5000,NaN,Q


In [58]:
df["FamiliSize"]=df["SibSp"]+df["Parch"]+1
df['Cabin']=df['Cabin'].fillna("Missing")
df["Deck"]=df["Cabin"].astype(str).str[0]
df.sample(6)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,FamiliSize,Deck
785,786,no,third,"Harmer, Mr. Abraham (David Lishin)",male,25.0,0,0,374887,7.2500,Missing,S,1,M
379,380,no,third,"Gustafsson, Mr. Karl Gideon",male,19.0,0,0,347069,7.7750,Missing,S,1,M
80,81,no,third,"Waelens, Mr. Achille",male,22.0,0,0,345767,9.0000,Missing,S,1,M
817,818,no,second,"Mallet, Mr. Albert",male,31.0,1,1,S.C./PARIS 2079,37.0042,Missing,C,3,M
768,769,no,third,"Moran, Mr. Daniel J",male,NaN,1,0,371110,24.1500,Missing,Q,2,M
46,47,no,third,"Lennon, Mr. Denis",male,NaN,1,0,370371,15.5000,Missing,Q,2,M


In [59]:
X=df.drop(["Survived"],axis=1)
y=df['Survived']

Xtrain,Xtest,ytrain,ytest=train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)

##outlier detection

In [60]:
meanAge=Xtrain["Age"].mean()
stdAge=Xtrain["Age"].std()
Xtrain["ZscoreAge"]=(Xtrain["Age"]-meanAge)/stdAge##zscore calculation

musk= (abs(Xtrain["ZscoreAge"])<=3)##to know in which row has less then 3 z score

Xtrain=Xtrain[musk]## to keep where the musk is true
ytrain=ytrain[musk]

Xtrain.shape
ytrain.shape

(573,)

In [61]:
fareQ1=Xtrain["Fare"].quantile(0.25)
fareQ3=Xtrain["Fare"].quantile(0.75)

IQRfare=fareQ3-fareQ1
minFare=max(0,fareQ1-1.5*IQRfare)##as fare cant be negative
maxFare=fareQ3+1.5*IQRfare

Xtrain["Fare"]=Xtrain["Fare"].clip(minFare,maxFare)


In [62]:
##numercal
p1=Pipeline(
    steps=[
        ("imputer",SimpleImputer(strategy="mean")),
        ("scaler",StandardScaler())
    ]
)
p2=Pipeline(
    steps=[
        ("imputer",SimpleImputer(strategy="median")),
        ("Scaler",MinMaxScaler())
    ]
)

In [63]:
##categorical
p3=Pipeline(
    steps=[
        ("imputer",SimpleImputer(strategy="most_frequent")),
        ("encoder",OneHotEncoder(sparse_output=False,drop="first",handle_unknown="ignore"))
    ]
)
p4=Pipeline(
    steps=[
        ("imputer",SimpleImputer(strategy="most_frequent")),
        ("encoder",OrdinalEncoder(categories=[["third","second","first"]])),
        ("Scaler",MinMaxScaler())
    ]

)

In [64]:
preprocessor=ColumnTransformer(
    transformers=[
        ("pipeline1",p1,["Age"]),
        ("pipeline2",p2,["Fare","FamiliSize"]),
        ("pipeline3",p3,["Embarked","Sex","Deck"]),
        ("pipeline4",p4,["Pclass"])
    ],
    remainder="drop"
)
preprocessor

ColumnTransformer(transformers=[('pipeline1',
                                 Pipeline(steps=[('imputer', SimpleImputer()),
                                                 ('scaler', StandardScaler())]),
                                 ['Age']),
                                ('pipeline2',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('Scaler', MinMaxScaler())]),
                                 ['Fare', 'FamiliSize']),
                                ('pipeline3',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('encoder',
                                                  OneHotEncoder(drop='first',
                                                                handle_unknown='ignore',
                                                                sparse_output=False))]),
                                 ['Embarked', 'Sex', 'Deck']),
                                ('pipeline4',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('encoder',
                                                  OrdinalEncoder(categories=[['third',
                                                                              'second',
                                                                              'first']])),
                                                 ('Scaler', MinMaxScaler())]),
                                 ['Pclass'])])

In [65]:
le=LabelEncoder()##encodin the ytrain though logistic regression can understand the categories

le.fit(ytrain)
ytrain=le.transform(ytrain)
ytest=le.transform(ytest)

In [66]:
lrModel=Pipeline(
    steps=[
        ("preprocessor",preprocessor),
        ("classifier",LogisticRegression(max_iter=1000,class_weight="balanced"))
    ]
)
lrModel

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('pipeline1',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer()),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['Age']),
                                                 ('pipeline2',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('Scaler',
                                                                   MinMaxScaler())]),
                                                  ['Fare', 'FamiliSize']),
                                                 ('pipeline3',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='m...
                                                                   OneHotEncoder(drop='first',
                                                                                 handle_unknown='ignore',
                                                                                 sparse_output=False))]),
                                                  ['Embarked', 'Sex', 'Deck']),
                                                 ('pipeline4',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OrdinalEncoder(categories=[['third',
                                                                                               'second',
                                                                                               'first']])),
                                                                  ('Scaler',
                                                                   MinMaxScaler())]),
                                                  ['Pclass'])])),
                ('classifier',
                 LogisticRegression(class_weight='balanced', max_iter=1000))])

In [67]:

Xtrain.drop(["PassengerId","Name","SibSp","Parch","ZscoreAge","Ticket"],axis=1,inplace=True)
Xtest.drop(["PassengerId","Name","SibSp","Parch","Ticket"],axis=1,inplace=True)

In [72]:
lrModel.fit(Xtrain,ytrain)
lrModel["classifier"].coef_##to see the weights and as it is a pipeline
lrModel["classifier"].intercept_##to see the bias
ypred=lrModel.predict(Xtest)

In [70]:
print(f"test sccuracy:{lrModel.score(Xtest,ytest)}")
print(f"train sccuracy:{lrModel.score(Xtrain,ytrain)}")

test sccuracy:0.7653631284916201
train sccuracy:0.7975567190226877


In [74]:
accuracy=round(accuracy_score(ytest,ypred),4)
precision=round(precision_score(ytest,ypred),4)
recall=round(recall_score(ytest,ypred),4)
f1=round(f1_score(ytest,ypred),4)
print(f"accuracy:{accuracy}")
print(f"precision:{precision}")
print(f"recall:{recall}")
print(f"f1:{f1}")

accuracy:0.7654
precision:0.6753
recall:0.7536
f1:0.7123
